# Momentum One — SIGON Run-All (Colab)

**Monty path:** Runtime → GPU → run cells top to bottom.

| Step | What |
|------|------|
| 0 | Runtime → Change runtime type → **L4 GPU** |
| 1 | Mount Drive, pull repo, copy CSVs, clear caches once |
| 2 | Train (leave running) |
| 3 | Jarvis (after Interrupt or after train ends) |

**Hard rules**
- Signals ON → new **~6820** brain. Never load PROVEN 1820 into SIGON.
- New all-time-high streak + breach **0%** → locks `best_sigon_streakXX.pt`
- Clear caches once after signals ON — never delete `.pt` brains

Full notes: `GPU_EDITION/COLAB_JARVIS_SIGON.md`

## Cell 0 — check GPU
If this says NONE: **Runtime → Change runtime type → L4 GPU → Save**, then re-run.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 GPU')

## Cell 1 — Drive + repo + CSVs + clear caches once

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/monty313/the-truth.git 2>/dev/null || true
%cd /content/the-truth
!git pull origin main
!pip -q install pyyaml >/dev/null 2>&1

import os, shutil, glob

candidates = [
    '/content/drive/MyDrive/Camillion_data',
    '/content/drive/MyDrive/the-truth-data',
    '/content/drive/MyDrive/MOMENTUM_ONE/02_PRICE_DATA',
    '/content/drive/MyDrive/1gwTD6535FilsKqRUsi7vi6WlT0vgdhui',
]
src = None
for c in candidates:
    if os.path.isdir(c) and glob.glob(c + '/**/*.csv', recursive=True):
        src = c
        break
if src is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        names = ' '.join(files).upper()
        if 'XAUUSD' in names and any(x in names for x in ('EURUSD', 'US30', 'GBPUSD')):
            if any(f.lower().endswith('.csv') for f in files):
                src = root
                break

os.makedirs('data', exist_ok=True)
if src:
    for f in glob.glob(src + '/**/*.csv', recursive=True):
        base = os.path.basename(f).upper()
        if any(s in base for s in ('XAUUSD', 'EURUSD', 'GBPUSD', 'US30')):
            dst = os.path.join('data', os.path.basename(f))
            shutil.copy2(f, dst)
            print('copied', dst)
else:
    print('WARNING: no CSVs on Drive — upload XAUUSD/EURUSD/GBPUSD/US30 into data/')

print('data CSVs:', sorted(os.listdir('data')) if os.path.isdir('data') else [])

# Clear feature caches ONCE after signals ON (never delete .pt brains)
!rm -f artifacts/gpu_cache_*.npz
!rm -rf artifacts/symbol_cache
print('caches cleared (gpu_cache + symbol_cache). checkpoints untouched.')
print('expect include_signal_agent_slots: true → obs_dim ~6820')

## Cell 2 — TRAIN (leave running)

**OOM?** change `--instances 8000` → `6000` → `4000` → `2000`.

Champion locks as:
- `artifacts/checkpoints/best_sigon.pt` (latest pointer)
- `artifacts/checkpoints/best_sigon_streak05.pt` (example: streak 5)
- `artifacts/checkpoints/history/SIGON_streak05_obs6820_SN-xxxx.pt`

Sampling: 40% @ 2.5%/3.5% · 60% random ranges · multi-symbol pool · 3 day-retries/instance.

In [ ]:
!python scripts/gpu_train.py --csv-dir data --instances 8000 --minutes 600 --entropy-coef 0.03 --warm best_sigon

## Cell 3 — JARVIS (after Interrupt, or after train ends)

Colab runs **one cell at a time**. Stop train (Interrupt) to run Jarvis, then re-run Cell 2 with the same command (`--warm best_sigon` resumes).

In [ ]:
!python scripts/jarvis_talk.py status
!python scripts/jarvis_talk.py board
print('--- progress ---')
!cat artifacts/checkpoints/gpu_progress.json 2>/dev/null || echo 'no progress yet'
print('--- champions ---')
!ls -1 artifacts/checkpoints/best_sigon*.pt 2>/dev/null || echo 'no champion yet'
!ls -1 artifacts/checkpoints/history/SIGON_streak*.pt 2>/dev/null || echo 'no history locks yet'

### Optional hot notes (writes inbox; applied next train loop)

```python
!python scripts/jarvis_talk.py "SET w_pullback_with_htf=0.35"
!python scripts/jarvis_talk.py "RELOAD_REWARDS"
!python scripts/jarvis_talk.py "NOTE dual HTF — take LTF pulls"
!python scripts/jarvis_talk.py outbox
```

## Cell 4 — optional Drive backup of champions

In [ ]:
import os, shutil, glob
dst = '/content/drive/MyDrive/momentum_sigon_champs'
os.makedirs(dst, exist_ok=True)
for p in glob.glob('artifacts/checkpoints/best_sigon*.pt'):
    shutil.copy2(p, os.path.join(dst, os.path.basename(p)))
    print('backed up', p)
print('done →', dst)